In [ ]:
# ============================================================
# Cell 1: Setup, Drive Mount & Folder Structure
# ============================================================
!pip install ultralytics roboflow -q

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
from datetime import datetime

# ── Folder Structure على Drive ────────────────────────────────
base_path      = '/content/drive/MyDrive/RoadEye_Japan_V1'
weights_path   = os.path.join(base_path, 'weights')
results_path   = os.path.join(base_path, 'results')
checkpoints    = os.path.join(base_path, 'checkpoints')

for folder in [base_path, weights_path, results_path, checkpoints]:
    os.makedirs(folder, exist_ok=True)

print("✅ Folder structure ready:")
print(f"   📁 {base_path}")
print(f"   ├── 📁 weights/       ← best.pt & last.pt")
print(f"   ├── 📁 results/       ← charts & metrics")
print(f"   └── 📁 checkpoints/   ← كل 10 epochs")

# GPU Check
import subprocess
gpu = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total',
     '--format=csv,noheader'],
    capture_output=True, text=True
).stdout.strip()
print(f"\n✅ GPU: {gpu}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 115.9 MB/s eta 0:00:00
Mounted at /content/drive
✅ Folder structure ready:
   📁 /content/drive/MyDrive/RoadEye_Japan_V1
   ├── 📁 weights/       ← best.pt & last.pt
   ├── 📁 results/       ← charts & metrics
   └── 📁 checkpoints/   ← كل 10 epochs

✅ GPU: Tesla T4, 15360 MiB


In [ ]:
# ============================================================
# Cell 2: Download Dataset
# ============================================================
from roboflow import Roboflow
import glob

rf      = Roboflow(api_key="RQMPMcpDeL2nFd04kTvI")
project = rf.workspace("roadeye-workspace").project("roadeye-japan-v1")
version = project.version(1)
dataset = version.download("yolov8")

yaml_files   = glob.glob(f"{dataset.location}/**/*.yaml", recursive=True)
dataset_yaml = yaml_files[0]

train_imgs = glob.glob(f"{dataset.location}/train/images/*")
val_imgs   = glob.glob(f"{dataset.location}/valid/images/*")

print(f"✅ YAML  : {dataset_yaml}")
print(f"✅ Train : {len(train_imgs)} images")
print(f"✅ Val   : {len(val_imgs)} images")

assert len(train_imgs) > 0, "❌ No training images"
assert len(val_imgs)   > 0, "❌ No validation images"

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to RoadEye-Japan-V1-1 in yolov8:: 100%|██████████| 6757/6757 [00:02<00:00, 2522.93it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ YAML  : /content/RoadEye-Japan-V1-1/data.yaml
✅ Train : 2532 images
✅ Val   : 506 images


In [ ]:
# ============================================================
# Cell 3: Training مع Auto-Save على Drive
# ============================================================
from ultralytics import YOLO
import shutil, os
from datetime import datetime

model = YOLO('yolov8m.pt')
print("✅ YOLOv8m loaded — Starting training...\n")

results = model.train(
    # ── Core ──────────────────────────────────────
    data         = dataset_yaml,
    imgsz        = 1024,
    epochs       = 150,
    batch        = 6,
    workers      = 2,

    # ── حفظ مباشرة على Drive ──────────────────────
    project      = base_path,
    name         = 'training_run',

    # ── Optimizer ─────────────────────────────────
    optimizer        = 'SGD',
    lr0              = 0.008,
    lrf              = 0.12,
    momentum         = 0.937,
    weight_decay     = 0.0005,
    warmup_epochs    = 3.0,
    warmup_momentum  = 0.8,
    warmup_bias_lr   = 0.1,
    cos_lr           = True,

    # ── Augmentation ──────────────────────────────
    mosaic       = 0.85,
    close_mosaic = 35,
    mixup        = 0.08,
    copy_paste   = 0.15,
    hsv_h        = 0.015,
    hsv_s        = 0.5,
    hsv_v        = 0.4,
    degrees      = 15.0,
    translate    = 0.15,
    scale        = 0.6,
    shear        = 4.0,
    perspective  = 0.0003,
    flipud       = 0.4,
    fliplr       = 0.5,

    # ── Loss ──────────────────────────────────────
    box          = 7.5,
    cls          = 0.6,
    dfl          = 1.5,

    # ── Regularization ────────────────────────────
    label_smoothing = 0.02,

    # ── System ────────────────────────────────────
    amp          = True,
    cache        = False,
    save         = True,
    save_period  = 10,   # checkpoint كل 10 epochs
    patience     = 35,
    val          = True,
    device       = 0,
)

# ============================================================
# بعد التدريب: نسخ كل حاجة مهمة على Drive بشكل منظم
# ============================================================
run_dir = os.path.join(base_path, 'training_run')

def safe_copy(src, dst):
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"   ✅ {os.path.basename(src)}")
    else:
        print(f"   ⚠️  Not found: {os.path.basename(src)}")

print("\n📦 Saving all outputs to Drive...")

# 1. Weights
print("\n🔵 Weights:")
safe_copy(f"{run_dir}/weights/best.pt",  f"{weights_path}/best.pt")
safe_copy(f"{run_dir}/weights/last.pt",  f"{weights_path}/last.pt")

# 2. Results charts
print("\n🔵 Charts & Metrics:")
for fname in ['results.png', 'confusion_matrix.png',
              'P_curve.png', 'R_curve.png',
              'PR_curve.png', 'F1_curve.png',
              'results.csv']:
    safe_copy(f"{run_dir}/{fname}", f"{results_path}/{fname}")

# 3. Checkpoints (كل الـ epoch weights)
print("\n🔵 Epoch Checkpoints:")
ckpt_src = os.path.join(run_dir, 'weights')
if os.path.exists(ckpt_src):
    for f in os.listdir(ckpt_src):
        if f.startswith('epoch'):
            safe_copy(f"{ckpt_src}/{f}", f"{checkpoints}/{f}")

# ── Final Report ──────────────────────────────────────────────
print("\n" + "="*55)
print("✅ TRAINING COMPLETE — ALL FILES SAVED TO DRIVE")
print("="*55)
print(f"mAP50    : {results.results_dict.get('metrics/mAP50(B)', 'N/A')}")
print(f"mAP50-95 : {results.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")
print(f"\n📁 Drive Location: {base_path}")
print(f"   ├── weights/best.pt          ← الموديل النهائي")
print(f"   ├── weights/last.pt          ← آخر epoch")
print(f"   ├── results/results.png      ← Training curves")
print(f"   └── checkpoints/epoch*.pt   ← كل 10 epochs")
print("="*55)

✅ YOLOv8m loaded — Starting training...

WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=35, cls=0.6, cls_pw=0.0, compile=False, conf=None, copy_paste=0.15, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/RoadEye-Japan-V1-1/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.4, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.008, lrf=0.12, mask_ratio=4, max_det=300, mixup=0.08, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=0

In [ ]:
!pip install ultralytics roboflow -q
# FINAL EVALUATION — شغّله دلوقتي
import shutil, os
from ultralytics import YOLO

base_path = '/content/drive/MyDrive/RoadEye_Japan_V1'

# ── 1. تأكد إن best.pt اتحفظ ─────────────────────────────────
best_src = f"{base_path}/training_run/weights/best.pt"
best_dst = f"{base_path}/weights/best_FINAL.pt"
shutil.copy2(best_src, best_dst)
print(f"✅ best.pt saved → {best_dst}")

# ── 2. Final Test Set Evaluation ─────────────────────────────
model   = YOLO(best_src)
metrics = model.val(
    data    = dataset_yaml,   # من Cell 2
    split   = 'test',         # على الـ test set مش val
    imgsz   = 1024,
    conf    = 0.40,
    iou     = 0.45,
    device  = 0,
    plots   = True,
    save_json = False,
)

# ── 3. النتائج ────────────────────────────────────────────────
print("\n" + "="*55)
print("ROADEYE — FINAL TEST SET RESULTS")
print("="*55)
print(f"  mAP50       : {metrics.box.map50:.4f}")
print(f"  mAP50-95    : {metrics.box.map:.4f}")
print(f"  Precision   : {metrics.box.mp:.4f}")
print(f"  Recall      : {metrics.box.mr:.4f}")
print(f"  Per-class mAP50:")
for i, (name, ap) in enumerate(zip(['crack','pothole'],
                                    metrics.box.ap50)):
    print(f"    {name:10s}: {ap:.4f}")
print("="*55)
print(f"\n✅ Model ready: {best_dst}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 98.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 108.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/RoadEye_Japan_V1/training_run/weights/best.pt'

In [ ]:
# Cell: دور على best.pt في كل الـ Drive
import glob, os

print("🔍 Searching for best.pt...")

# دور في كل المسارات الممكنة
search_paths = [
    '/content/drive/MyDrive/**/*.pt',
    '/content/drive/.shortcut-targets-by-id/**/*.pt',
]

found = []
for pattern in search_paths:
    results = glob.glob(pattern, recursive=True)
    found.extend(results)

if found:
    for f in found:
        size = os.path.getsize(f) / 1024 / 1024
        print(f"  ✅ {f}")
        print(f"     Size: {size:.1f} MB")
else:
    print("  ❌ No .pt files found")

🔍 Searching for best.pt...
  ❌ No .pt files found
